# Proof of Concept: Webscraping-News + eigene Sentiment-Analyse + Forex-Vergleich

Dieses Notebook ist der **Proof of Concept (PoC)** zum sauberen Weg (EODHD-News + Yahoo/EODHD-Forex + Oel).

**Ziel:** Die Methodik des sauberen Weges (siehe `news_forex_korrelation_kombiniert.ipynb`) auf eine **zweite, unabhaengige Nachrichtenquelle** uebertragen und damit pruefen, ob der gefundene Zusammenhang reproduzierbar ist.

**Einschraenkung:** Webscraping (RSS + Reddit) liefert nur die **juengste Geschichte** der Feeds. Der Ueberlapp mit Forex-Daten ist kurz (wenige Wochen), weshalb dies strikt ein **PoC** ist, **keine** statistisch belastbare Analyse.

## Schritte
1. Setup und Daten laden (alle Scrapes aus `data/raw/news/webscraping/`)
2. Kombination + Duplikat-Kontrolle + Datenqualitaet
3. Eigene Sentiment-Analyse mit TextBlob (gleiche Methodik wie `sentiment_analyse_vergleich.ipynb`)
4. Aggregation auf Tagesbasis (Median pro Tag, analog zum sauberen Weg)
5. Forex-Daten (Yahoo+EODHD kombiniert) laden und Ueberlapp bestimmen
6. Korrelation und visueller Vergleich
7. Persistierung und Diskussion der Limitierungen

**Quelle fuer Methodik-Entscheidungen:** Slides zum Kurs (vor allem Imputation, Aggregation) und `CLAUDE.md` des Projekts.

## 1. Setup und Daten laden

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob

plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (14, 6)
pd.set_option("display.max_columns", None)

WEB_DIR = os.path.join("..", "..", "data", "raw", "news", "webscraping")
FOREX_COMBINED = os.path.join("..", "..", "data", "processed", "forex", "forex_alle_quellen_kombiniert.csv")
PROCESSED_NEWS = os.path.join("..", "..", "data", "processed", "news")
os.makedirs(PROCESSED_NEWS, exist_ok=True)

print("Setup erfolgreich!")

In [ ]:
# Alle Webscraping-Dateien vom Typ all_scraped_news_*.csv einlesen.
# Jede Datei ist ein Schnappschuss zu einem Scrape-Zeitpunkt.
# Wichtig: Dateien mit Suffix wie _PRE-FIX werden ignoriert (das waren
# fehlgeschlagene Laeufe, die bewusst nicht in die Analyse einfliessen).

files = sorted(glob.glob(os.path.join(WEB_DIR, "all_scraped_news_*.csv")))
files = [f for f in files if "PRE-FIX" not in f]

print(f"Gefundene Scrape-Dateien: {len(files)}")
dfs = []
for f in files:
    df = pd.read_csv(f)
    df["scrape_file"] = os.path.basename(f)
    print(f"  {os.path.basename(f):40s} {len(df):5d} Zeilen")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce", utc=True)
df_all["date_only"] = pd.to_datetime(df_all["date_only"], errors="coerce")

print(f"\nGesamt vor Deduplikation: {len(df_all)} Eintraege aus {len(files)} Schnappschuessen")

## 2. Duplikat-Kontrolle und Datenqualitaet

**Warum Duplikate?** RSS-Feeds zeigen immer die letzten ~20-50 Artikel. Ein Artikel der am 03.03. erscheint, ist am 04.03. moeglicherweise noch im Feed und wird erneut gescrapt. Dedup-Schluessel: `link` (eindeutig pro Artikel).

**Was waere falsch?** Einfach alle Eintraege behalten → gleicher Artikel wuerde mehrfach in Sentiment-Aggregation einfliessen und das Tagesmedian verzerren.

In [ ]:
before = len(df_all)
dupe_link = df_all.duplicated(subset="link").sum()
dupe_title = df_all.duplicated(subset="title").sum()
print(f"Duplikate nach link:  {dupe_link}")
print(f"Duplikate nach title: {dupe_title}")
print(f"Leere Titel:          {(df_all['title'].fillna('').str.strip() == '').sum()}")
print(f"Leere Links:          {(df_all['link'].fillna('').str.strip() == '').sum()}")
print(f"Ungueltige Datumswerte: {df_all['date'].isna().sum()}")

# Deduplizieren nach link, haerten gegen leere Links
df_unique = df_all.drop_duplicates(subset="link", keep="first").reset_index(drop=True)
print(f"\nNach Dedup auf link: {len(df_unique)} unique Eintraege (entfernt: {before - len(df_unique)})")

In [ ]:
# Datenqualitaet: Verteilung pro Quelle und Zeitraum
print("Eintraege pro Quelle:")
print(df_unique["source"].value_counts())

valid_dates = df_unique["date"].dropna()
print(f"\nZeitraum: {valid_dates.min()} bis {valid_dates.max()}")
print(f"Artikel ohne gueltiges Datum: {df_unique['date'].isna().sum()} "
      f"({df_unique['date'].isna().mean()*100:.1f}%)")

# Monatliche Verteilung (nur mit gueltigem Datum)
df_unique["month"] = df_unique["date"].dt.to_period("M")
monthly = df_unique.dropna(subset=["date"]).groupby("month").size()
print(f"\nArtikel pro Monat:")
print(monthly)

## 3. Eigene Sentiment-Analyse (TextBlob)

**Methodik:** Gleiche Methodik wie in `sentiment_analyse_vergleich.ipynb` fuer EODHD-Text, damit beide Quellen vergleichbar sind. TextBlob liefert `polarity` in [-1, 1] (-1 = sehr negativ, +1 = sehr positiv) und `subjectivity` in [0, 1].

**Was wird analysiert?** Titel + Summary. Fuer Reddit-Posts ist der Summary-Text (selftext) meist leer oder sehr kurz; der Titel traegt das meiste Signal.

In [ ]:
def compute_sentiment(row: pd.Series) -> pd.Series:
    title = str(row.get("title") or "")
    summary = str(row.get("summary") or "")
    text = (title + ". " + summary).strip()
    if not text or text == ".":
        return pd.Series({"polarity_tb": np.nan, "subjectivity_tb": np.nan})
    blob = TextBlob(text)
    return pd.Series({
        "polarity_tb": blob.sentiment.polarity,
        "subjectivity_tb": blob.sentiment.subjectivity,
    })

sent = df_unique.apply(compute_sentiment, axis=1)
df_unique = pd.concat([df_unique, sent], axis=1)

print("Sentiment-Verteilung (polarity_tb):")
print(df_unique["polarity_tb"].describe())
print(f"\nAnteil neutraler Artikel (polarity == 0): "
      f"{(df_unique['polarity_tb'] == 0).mean()*100:.1f}%")

In [ ]:
# Verteilung der Polarity und per Quelle
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(df_unique["polarity_tb"].dropna(), bins=50, edgecolor="black", color="steelblue")
axes[0].axvline(0, color="red", linestyle="--", alpha=0.4)
axes[0].set_title("Verteilung der TextBlob-Polarity (alle Artikel)")
axes[0].set_xlabel("polarity")
axes[0].set_ylabel("Anzahl")

sns.boxplot(data=df_unique, x="source", y="polarity_tb", ax=axes[1])
axes[1].axhline(0, color="red", linestyle="--", alpha=0.4)
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("Polarity pro Quelle")
plt.tight_layout()
plt.show()

## 4. Aggregation: Tagesmedian der Polarity

**Warum Median?** Robust gegen einzelne extreme Artikel (z.B. ein Artikel mit polarity=1.0 zieht das Tagesmittel stark nach oben). Gleiche Aggregations-Regel wie im sauberen Weg (`news_forex_korrelation_kombiniert.ipynb`).

**Was wird verworfen?** Artikel ohne gueltiges Datum oder ohne berechenbare Polarity.

In [ ]:
df_sent = df_unique.dropna(subset=["date", "polarity_tb"]).copy()
df_sent["date_only"] = df_sent["date"].dt.tz_convert(None).dt.normalize()

daily_sentiment = (
    df_sent.groupby("date_only")
    .agg(polarity_median=("polarity_tb", "median"),
         polarity_mean=("polarity_tb", "mean"),
         n_articles=("polarity_tb", "size"))
    .sort_index()
)
daily_sentiment.index.name = "date"
print(f"Aggregierte Tage: {len(daily_sentiment)}")
print(f"Zeitraum: {daily_sentiment.index.min().date()} bis {daily_sentiment.index.max().date()}")
daily_sentiment.head()

## 5. Forex-Daten laden und Ueberlapp bestimmen

Wir laden die kombinierte Forex-Datei (Yahoo + EODHD, aus `datenanalyse_forex.ipynb`) und schneiden sie auf den Ueberlapp mit dem Webscraping-Sentiment-Zeitraum zu.

In [ ]:
df_forex = pd.read_csv(FOREX_COMBINED, index_col="date", parse_dates=True)
print(f"Forex-Datei: {len(df_forex)} Zeilen")

# Fuer jedes Paar: Close-Mittel aus yahoo+eodhd (nur wo beide existieren)
pairs = ["EUR_USD", "EUR_CHF", "GBP_USD"]
close_per_pair = {}
for p in pairs:
    sub = df_forex[df_forex["pair"] == p]
    yh = sub.get("yahoo_close")
    ed = sub.get("eodhd_close")
    mean_close = pd.concat([yh, ed], axis=1).mean(axis=1, skipna=False).dropna()
    close_per_pair[p] = mean_close
    print(f"  {p}: {len(mean_close)} Tage, {mean_close.index.min().date()} bis {mean_close.index.max().date()}")

# Ueberlapp mit Sentiment
sent_days = set(daily_sentiment.index.normalize())
for p in pairs:
    fx_days = set(close_per_pair[p].index.normalize())
    overlap = sent_days & fx_days
    print(f"\n  Ueberlapp Sentiment x {p}: {len(overlap)} Tage")

## 6. Korrelation und visueller Vergleich

**Pearson auf taeglichen Werten** ist auf so kurzem Fenster selten belastbar. Wir berichten deshalb zusaetzlich Wochen-Aggregation (Mittel pro Woche) zur optischen Plausibilitaetsprüfung.

In [ ]:
for p in pairs:
    fx = close_per_pair[p]
    df_join = daily_sentiment[["polarity_median", "n_articles"]].join(fx.rename("close"), how="inner")
    if df_join.empty:
        print(f"{p}: Kein Ueberlapp, ueberspringe.")
        continue

    # Tages-Returns vs. Sentiment (Pearson)
    df_join["ret"] = df_join["close"].pct_change()
    corr_level = df_join["polarity_median"].corr(df_join["close"])
    corr_ret = df_join["polarity_median"].corr(df_join["ret"])
    print(f"{p}: n_tage={len(df_join)}  corr(sent, close)={corr_level:+.3f}  "
          f"corr(sent, ret)={corr_ret:+.3f}")

In [ ]:
fig, axes = plt.subplots(len(pairs), 1, figsize=(14, 4 * len(pairs)))
for ax, p in zip(axes, pairs):
    fx = close_per_pair[p]
    df_join = daily_sentiment[["polarity_median"]].join(fx.rename("close"), how="inner")
    if df_join.empty:
        ax.set_title(f"{p} – kein Ueberlapp")
        continue

    ax2 = ax.twinx()
    ax.plot(df_join.index, df_join["close"], color="#1f77b4", label=f"{p} close")
    ax2.plot(df_join.index, df_join["polarity_median"], color="#d62728", linestyle=":", label="Polarity (Median)")
    ax2.axhline(0, color="gray", linestyle="--", alpha=0.3)
    ax.set_ylabel(f"{p} close", color="#1f77b4")
    ax2.set_ylabel("Polarity", color="#d62728")
    ax.set_title(f"{p}: Forex vs. Webscraping-Sentiment (n={len(df_join)} Tage)")
plt.tight_layout()
plt.show()

## 7. Persistierung

Die bereinigten + sentiment-annotierten Webscraping-Daten und die tagesweise aggregierte Polarity werden nach `data/processed/news/` geschrieben.

In [ ]:
# Artikel-Level-Output (ein Artikel pro Zeile, mit Sentiment)
cols_out = ["date", "date_only", "source", "title", "summary", "link",
             "polarity_tb", "subjectivity_tb", "scrape_file"]
cols_out = [c for c in cols_out if c in df_unique.columns]
out_articles = os.path.join(PROCESSED_NEWS, "webscraping_articles_sentiment.csv")
df_unique[cols_out].to_csv(out_articles, index=False)
print(f"Gespeichert: {out_articles} ({len(df_unique)} Artikel)")

# Tagesaggregat
out_daily = os.path.join(PROCESSED_NEWS, "webscraping_sentiment_daily.csv")
daily_sentiment.to_csv(out_daily)
print(f"Gespeichert: {out_daily} ({len(daily_sentiment)} Tage)")

## 8. Limitierungen & Diskussion

- **Abdeckung:** Webscraping liefert nur jene Artikel, die zum Scrape-Zeitpunkt im Feed sichtbar waren. Historische Artikel aus den Feeds gehen verloren. Der Ueberlapp mit Forex-Daten ist dadurch systematisch auf die letzten Wochen vor den Scrapes beschraenkt.
- **Quellenmix:** RSS (Finanznachrichten, meist neutral) und Reddit (oft meinungsstark) mischen sich; Reddit zieht die Polarity-Verteilung in beide Richtungen. Fuer eine spaetere Version sollte man Quellen gewichten oder nur RSS verwenden.
- **TextBlob:** Einfaches lexikonbasiertes Verfahren; liefert fuer Finanztexte regelmaessig 0-Werte (neutral), wenn kein Signal erkannt wird. Das ist erwartet und entspricht den Erfahrungen aus `sentiment_analyse_vergleich.ipynb`.
- **Kurzes Zeitfenster:** Pearson-Korrelationen auf <30 Tagen sind instabil und muessen mit Vorsicht gelesen werden. Dieses Notebook ist explizit ein PoC, keine Validierung.
- **Feiertage/Wochenenden:** Wenn kein Artikel-Median berechenbar ist, bleibt die Reihe NaN – nicht imputieren (gleiches Prinzip wie beim sauberen Weg mit EODHD).

**Naechste Schritte (optional):** Feed-Gewichtung, Forex-Returns als Zielvariable statt Level, Lag-Korrelation (Sentiment heute → Return morgen), Vergleich mit der EODHD-basierten Polarity aus dem sauberen Weg (liegt als `data/processed/news/webscraping_sentiment_daily.csv` bereit).